# Asari & Zador (2009), Figure 4: STRF and LN models of Wehr's recordings (CRCNS ac-1)

**Paper.** Asari H, Zador AM (2009). Long-lasting context dependence constrains neural encoding
models in rodent auditory cortex. *J Neurophysiol* 102:2638-2656.
[doi:10.1152/jn.00577.2009](https://doi.org/10.1152/jn.00577.2009). The claims file
(`claims/asari-2009.yaml`) was written from the version of record.

**Data set.** CRCNS ac-1, [doi:10.6080/K0KW5CXR](https://doi.org/10.6080/K0KW5CXR), version
`unversioned` (CRCNS does not version data sets; `reproduce_asari_2009.ipynb` prints the published
MD5 sums). This notebook reads `crcns-ac1.zip`, folder `wehr/`: Michael Wehr's whole-cell
recordings of Machens et al. (2004), one file per trial, with the stimuli and the analysis code
(`wehr/Tools/STRFanalysis.m`).

**Claim tested here.** fig4-strf-performance-low. The other claims are tested in
`reproduce_asari_2009.ipynb`.

**Selection, an assumption.** The paper "(re)analyzed the recording data (20 cells) from the
previous work (Machens et al. 2004)" (Methods, Neural encoding models) and does not list them. The
data set holds 25 cells; Eq. 52, the predictable power behind the performance measure, needs at
least two trials of a stimulus. Every cell with at least one natural sound presented at least twice
is used, with those stimuli only. The method details are listed in `wehr_strf.py`.

In [ ]:
import time
import resource
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from scipy import stats

import wehr_strf as W

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 300)
started = time.time()
rows = pd.DataFrame(W.summary())
rows["natural"] = rows.description.map(W.is_natural)

## Data check

In [ ]:
print("Data set:", W.DATASET, "(ac-1, version unversioned), read from", W.asari.BUCKET + W.asari.MAIN_ZIP)
print("wehr/Summary.mat groups:", rows.groupby("group").cell.nunique().to_dict(), "; files:", len(rows))
per = rows[rows.natural].groupby(["group", "cell", "stimulus"]).size().rename("trials").reset_index()
cells = per.groupby(["group", "cell"]).agg(natural_stimuli=("stimulus", "size"), with_2_trials=("trials", lambda t: int((t >= 2).sum())),
                                            trials_min=("trials", "min"), trials_max=("trials", "max")).reset_index()
print(cells.to_string(index=False))
use = sorted(set(cells[cells.with_2_trials > 0].cell))
print("Cells listed in more than one group:", sorted(cells.cell[cells.cell.duplicated()].unique()))
print(f"\nCells with at least one natural sound presented at least twice: {len(use)} of {len(cells)} (paper: 20 cells, not listed)")

In [ ]:
t0 = time.time()
data = {}
for c in use:
    stims, counts = W.load_cell(rows[rows.cell == c].to_dict("records"))
    data[c] = stims
print(f"Trials loaded in {(time.time() - t0) / 60:.1f} min")
chk = W.load_trial(rows[(rows.cell == use[0]) & rows.natural].file.iloc[0])
print("Response sampling rate of a trial file:", chk["sf"], "Hz; responses are averaged in", W.DT_MS, "ms bins")
ys = np.concatenate([R.ravel() for d in data.values() for R in d.values()])
print(f"Binned, centred responses (mV): 1st to 99th percentile {np.percentile(ys, 1):.2f} to {np.percentile(ys, 99):.2f}")
summ = pd.DataFrame([{"cell": c, "stimuli": len(d), "trials_per_stimulus": sorted({R.shape[0] for R in d.values()}),
                      "seconds": sum(R.shape[1] for R in d.values()) * W.DT_MS / 1000} for c, d in data.items()])
print(summ.to_string(index=False))
print("Each trial file's trigger is at its first sample (the trace starts at the sound onset), as Contents.m and ViewResponse.m use it.")

## fig4-strf-performance-low

> Linear STRF models of 20 A1 cells from Machens et al. (2004) predicted less than about 20% of the
> predictable response power, and extending the window from 180 ms to about 4 s did not improve them
> significantly, with or without a static nonlinearity; the LN model did only slightly better.

Per cell and window (180 to 4052 ms, 765 parameters each): ridge regression with 10-fold
cross-validation, the static nonlinearity by robust lowess on the training set, and the performance
of Eq. 53 on the validation (lower) and training (upper) sets. Readings: Hann window of 10 ms
(main) or 20 ms, folds as contiguous blocks (main) or whole stimuli (with the 10-ms window). The
test of "did not improve significantly" is the claims file's: a one-sided Wilcoxon signed-rank test
over cells of validation performance at 4052 against 180 ms, for STRF and LN, at 0.05; a Friedman
test over the nine windows is a variant.

In [ ]:
READ = [("hann 10 ms", 10, ("contiguous", "stimulus")), ("hann 20 ms", 20, ("contiguous",))]
def run(c):
    out = []
    for name, win, schemes in READ:
        specs, n_pads = W.cell_specs(data[c], win)
        for r in W.fit_cell(data[c], specs, n_pads, schemes=schemes):
            out.append({"cell": c, "window_fn": name, **r})
    return out
t0 = time.time()
with ThreadPoolExecutor(3) as ex:
    res = pd.DataFrame([r for o in ex.map(run, use) for r in o])
print(f"Models fitted in {(time.time() - t0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")
print("Fits whose best ridge value is at an end of the grid:", int((~res.ridge_interior).sum()), "of", len(res),
      "; by cell:", res[~res.ridge_interior].groupby("cell").size().to_dict())

In [ ]:
cols = ["strf_lower", "strf_upper", "ln_lower", "ln_upper"]
mean = res.groupby(["window_fn", "folds", "window_ms"])[cols].mean().round(3)
print("Mean performance over cells, by reading and window:")
print(mean.to_string())
rows_t = []
for (wf, fo), g in res.groupby(["window_fn", "folds"]):
    a, b = g[g.window_ms == 180].set_index("cell"), g[g.window_ms == 4052].set_index("cell")
    row = {"window_fn": wf, "folds": fo, "cells": len(a)}
    for m in ("strf", "ln"):
        p = stats.wilcoxon(b[m + "_lower"], a[m + "_lower"], alternative="greater").pvalue
        piv = g.pivot(index="cell", columns="window_ms", values=m + "_lower")
        row[m + "_wilcoxon_p"] = p
        row[m + "_change_4052_vs_180"] = float((b[m + "_lower"] - a[m + "_lower"]).mean())
        row[m + "_friedman_p"] = stats.friedmanchisquare(*[piv[w] for w in piv.columns]).pvalue
    mm = mean.loc[(wf, fo)]
    row["max_mean_lower"] = float(mm[["strf_lower", "ln_lower"]].max().max())
    row["max_mean_upper"] = float(mm[["strf_upper", "ln_upper"]].max().max())
    row["ln_gain_lower_mean"] = float((mm.ln_lower - mm.strf_lower).mean())
    rows_t.append(row)
tests = pd.DataFrame(rows_t)
print("\nPer reading: one-sided Wilcoxon (4052 > 180 ms, validation) and Friedman over windows; largest mean performance over windows:")
print(tests.round(4).to_string(index=False))
for _, r in tests.iterrows():
    for est in ("lower", "upper"):
        ok = r["max_mean_" + est] <= 0.22 and r.strf_wilcoxon_p >= 0.05 and r.ln_wilcoxon_p >= 0.05
        print(f"  {r.window_fn}, {r.folds} folds, {est} estimate: every window at most 0.22 for STRF and LN: {r['max_mean_' + est] <= 0.22}; "
              f"no significant improvement for either: {r.strf_wilcoxon_p >= 0.05 and r.ln_wilcoxon_p >= 0.05}; criterion met: {ok}")

In [ ]:
main = res[(res.window_fn == "hann 10 ms") & (res.folds == "contiguous")]
print("Main reading, per cell (validation performance at 180 and 4052 ms, STRF and LN):")
ps = main.pivot(index="cell", columns="window_ms", values="strf_lower")[[180, 4052]].add_suffix(" ms STRF")
pl = main.pivot(index="cell", columns="window_ms", values="ln_lower")[[180, 4052]].add_suffix(" ms LN")
print(pd.concat([ps, pl], axis=1).round(3).to_string())
print("Range of every plotted or tabled performance:", round(res[cols].min().min(), 3), "to", round(res[cols].max().max(), 3))
odd = "20020602-mw-001"
rest = res[res.cell != odd]
print(f"\nWithout {odd} (LN validation performance -0.574 at 180 ms and -0.713 at 4052 ms), {rest.cell.nunique()} cells:")
mr = rest.groupby(["window_fn", "folds", "window_ms"])[["strf_lower", "ln_lower"]].mean()
print(mr.round(3).unstack("window_ms").to_string())
for (wf, fo), g in rest.groupby(["window_fn", "folds"]):
    a, b = g[g.window_ms == 180].set_index("cell"), g[g.window_ms == 4052].set_index("cell")
    mm = mr.loc[(wf, fo)]
    print(f"  {wf}, {fo} folds: largest mean validation performance {float(mm.max().max()):.3f}; mean LN gain over STRF {float((mm.ln_lower - mm.strf_lower).mean()):.3f}; "
          f"Wilcoxon 4052 > 180 ms p = {stats.wilcoxon(b.strf_lower, a.strf_lower, alternative='greater').pvalue:.4f} (STRF), "
          f"{stats.wilcoxon(b.ln_lower, a.ln_lower, alternative='greater').pvalue:.4f} (LN)")


**Result: not reproduced.** Performance is low: in the main reading (10-ms Hann window, contiguous folds) the mean validation performance over the 16 cells is 0.178 to 0.207 for STRF and 0.165 to 0.209 for LN across the nine windows, at most 0.22 as the criterion requires (training performance 0.232 to 0.388). But extending the window from 180 to 4052 ms improves the validation performance a little and, by the claims file's test, significantly: by 0.030 for STRF (one-sided Wilcoxon p = 0.011) and 0.025 for LN (p = 0.042); the 20-ms window gives the same (p = 0.0065 and 0.019). With whole stimuli as folds the improvement is not significant (p = 0.12 and 0.088), but the mean LN validation performance reaches 0.241 at 3540 ms, above 0.22. So no reading meets both parts of the criterion. The LN model is not better than the STRF on average (mean gain -0.010 in the main reading). One cell (20020602-mw-001) has no predictive linear component (STRF performance near 0, LN -0.574 at 180 ms and -0.713 at 4052 ms, best ridge at the end of the grid in 22 of its 27 fits), and two of these statements rest on it. Without it (15 cells, main reading) the LN model is better than the STRF by 0.025 on average, and the mean validation performance reaches 0.222 (STRF) and 0.253 (LN), above 0.22; the longer window still helps significantly (p = 0.011 for STRF and 0.0075 for LN), so the outcome is the same. The outcome rests on the cell selection (16 cells with repeated stimuli against the paper's 20) and the claims file's test.

## Summary

| Claim | Outcome |
|---|---|
| fig4-strf-performance-low | not reproduced (assumed cell selection and test) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The paper's 20 cells are not listed; the 16 cells
with repeated natural sounds are used, with those sounds only. The spectrogram floor, the ridge
grid, the lowess evaluated on 60 points, and the test of "did not improve significantly" (the claims
file's one-sided Wilcoxon test) are choices the paper does not fix. The 10-kHz response reading of
the claims file does not apply: the responses are sampled at 4 kHz, and 4-ms bins are used.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")